# Train DustyLM from Scratch

Train Dusty, a tiny ~8M parameter assistant that talks like a robot vacuum. From raw data to a chatting model.

**What you will do:**
1. Set up the environment (Colab, RunPod, or local)
2. Download the training datasets
3. Train the tokenizer
4. Prepare tokenized data
5. Pretrain the base model
6. Select the best base checkpoint
7. Fine-tune with SFT
8. Select the best SFT checkpoint
9. Chat with your trained model

## Setup

**Google Colab (recommended):**

1. Open **Runtime > Change runtime type**.
2. Select **T4 GPU**, then click **Save**.
3. Run the Setup cell below, then continue through the notebook.

The default path takes about 15 minutes on a free Colab T4. Pretraining takes most of that time; SFT is much faster.

**RunPod:** Start a GPU pod, then run the Setup cell below.

**Local laptop:**

1. If needed, [install uv](https://docs.astral.sh/uv/getting-started/installation/), then run `uv sync` from the repository root.
2. Select the project's `.venv` as your notebook kernel.
3. Run the Setup cell below.


In [ ]:
import os
import sys
from pathlib import Path
import torch

is_cloud = "google.colab" in sys.modules or "RUNPOD_POD_ID" in os.environ
if is_cloud:
    !pip install -q uv
    if not Path("Makefile").exists():
        !git clone --depth 1 https://github.com/khordoo/dusty-lm.git
        os.chdir("dusty-lm")
    !pip install -q -e .
elif not Path("Makefile").exists():
    # Walk up so make commands find the Makefile from any CWD
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "Makefile").exists():
            os.chdir(str(parent))
            break
    else:
        print("Run this from the repo root:\n        uv sync")
REPO_ROOT = Path.cwd()
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print("✅ Setup complete. Repo root:", REPO_ROOT)
print("PyTorch:", torch.__version__)
print("Training device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

<details>
<summary>Windows users</summary>

This notebook uses `make` to keep commands short. On Colab, RunPod, macOS, Linux, and WSL, those commands should work as written.

On native Windows, use the commented `uv run python ...` command shown under each main `make` cell. If you prefer to run the `make` commands directly, use WSL or install `make` with a package manager such as Chocolatey or Scoop.

</details>

---
## 1. Download the Datasets

Downloads TinyStories for pretraining and the Dusty SFT JSONL from [mkhordoo/dusty-chat](https://huggingface.co/datasets/mkhordoo/dusty-chat). If you already have `artifacts/datasets/tinystories_base.txt` and `artifacts/datasets/dusty_sft.jsonl`, skip this cell.

**Why TinyStories?**
The sole purpose of pre-training is to teach basic English grammar and logic. Because our custom tokenizer is intentionally constrained to 4,096 tokens, we need simplified language. The Dusty vacuum persona naturally aligns with the direct, limited vocabulary of a 3-to-4-year-old. TinyStories fits these constraints perfectly, letting the 8M model learn correct English structure without being overwhelmed by complex syntax.

By default this downloads a 100k subset of TinyStories.

<details>
<summary>Adjust dataset size</summary>

If you have more system RAM, you can increase the slice size:

`!make download-datasets TINYSTORIES_SLICE="train[:200000]"`

or:

`!make download-datasets TINYSTORIES_SLICE="train[:500000]"`

Free Colab can be tight above 100k, so only use larger slices if your runtime has enough system RAM. To download the full dataset, pass `TINYSTORIES_SLICE="train"`.

</details>

In [ ]:
!make download-datasets
# Windows fallback, or if you want to see what make runs under the hood:
# !uv run python data_pipeline/download_datasets.py

## 2. Check the Training Files

Training needs TinyStories text and SFT data under `artifacts/datasets/`. This cell verifies the required files exist.

In [ ]:
required_sources = {
    "pretrain text": REPO_ROOT / "artifacts" / "datasets" / "tinystories_base.txt",
    "SFT chat JSONL": REPO_ROOT / "artifacts" / "datasets" / "dusty_sft.jsonl",
}

for label, path in required_sources.items():
    status = "✓ found" if path.exists() else "✗ missing"
    print(f"{label:16} {status:10} {path}")

---
## 3. Train the Tokenizer

Before Dusty can learn, it needs to know how to read. Standard tokenizers (like OpenAI's `tiktoken`) use massive vocabularies that would completely overwhelm our tiny model's 8M parameter capacity. Since Dusty only needs to understand simple, child-like English and basic vacuum commands, we will train a custom, highly efficient 4,096-token BPE tokenizer from scratch.

This step trains the tokenizer on our local text and saves it to `artifacts/tokenizers/dusty_tokenizer.json`.

> **🚨 Important:** Tokenizer training happens **only** at the very beginning of the pre-training phase. You must **never** retrain the tokenizer before or during the SFT phase. The base model's weights are permanently mapped to specific token IDs; retraining shuffles these IDs, causing the model to output pure garbage. Treat the tokenizer as a strictly fixed artifact for the entire remaining pipeline.


In [ ]:
!make tokenizer
# Windows fallback, or if you want to see what make runs under the hood:
# !uv run python -m dustylm.tokenizer
print("✅ Tokenizer successfully trained! Vocabulary is now locked at 4,096 tokens.")

## 4. Prepare Tokenized Datasets

Turns raw text and chat JSONL into Hugging Face `datasets` saved on disk. The training loop reads these tokenized datasets directly. This takes about 2 minutes.

In [ ]:
!make data-pretrain
!make data-sft
# Windows fallback, or if you want to see what make runs under the hood:
# !uv run python -m dustylm.data_prep --profile dusty8m
# !uv run python -m dustylm.data_prep --profile sft_dusty8m
print(
    "✅ Data preparation complete! Both pre-training and SFT datasets are successfully tokenized, packed, and ready for the training loop."
)

---
## 5. Pre-train the Base Model

Pretraining teaches the base `dusty8m` model language patterns before SFT teaches it to answer user messages. This notebook trains one epoch on a 100K TinyStories slice to keep the complete pipeline fast and practical on free Colab hardware. Even at this reduced scale, the model learns basic English structure and generates simple, coherent text, so you finish with a working base model.

> **Training longer?** Repeating the same 100K slice is not equivalent to adding more diverse data and can increase memorization. The [Advanced Tools notebook](03_advanced_tools.ipynb) covers data scale, training tokens, and inference tradeoffs in more detail.

**Start TensorBoard first** (non-blocking, runs in background), then start training.


### 5.1 Monitor Training with TensorBoard

The `%tensorboard` magic is non-blocking: it starts the dashboard as a background process and renders the UI immediately. You can run the training cell right after without waiting.

TensorBoard auto-refreshes every 30 seconds. The chart will be empty until training starts producing metrics. Click the 🔄 refresh icon once training is running to see the loss curve.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir artifacts/tensorboard



### 5.2 Start Pre-training
Now we will train the foundational English grammar for one epoch (about 382 steps).

⏱️ **Expected Runtime:** ~9 to 11 minutes (on a standard Google Colab T4 GPU). Grab a coffee!

In [ ]:
%%time
# Colab T4 can run batch 224 comfortably.
# Reduce BATCH_SIZE=32 if running locally on an older laptop.
EPOCHS = 1
!make train-pretrain EPOCHS={EPOCHS} CHECKPOINT_EVERY_STEPS=50 BATCH_SIZE=224
# Windows fallback, or if you want to see what make runs under the hood:
# !uv run python -m dustylm.train --profile dusty8m --epochs {EPOCHS} --checkpoint-every-steps 50 --batch-size 224
print(
    f"✅ Pre-training complete! The {EPOCHS}-epoch base model and all step checkpoints have been successfully saved to the artifacts folder."
)

### 5.3 The Full Training Lifecycle Loss Curve

Below is the TensorBoard loss curve from our production runs, showing both phases of training overlaid:

* **Green Line (Pre-training):** Runs for ~400 steps (1 epoch on 100K TinyStories). The loss drops sharply as the model learns basic English grammar.
* **Purple Line (SFT):** Runs for ~300 steps (2 epochs on the SFT dataset). The loss starts much lower because the model already understands language; SFT just steers the persona.

If you are running this locally, spin up TensorBoard to ensure your pre-training curve follows a similar downward trajectory before moving to SFT.

<img src="../docs/images/training-lifecycle-loss.png" alt="Dusty pretraining loss curve" width="600"/>


## 6. Check the Pretrained Base Model

Before SFT, generate from the `dusty8m` base profile. This tells you whether the base checkpoint has learned stable text patterns.

In [ ]:
!make generate PROFILE=dusty8m PROMPT="Once upon a time" TOP_P=0.9 TEMPERATURE=0.8
# Windows fallback, or if you want to see what make runs under the hood:
# !uv run python dustylm/generate.py --profile dusty8m --prompt "Once upon a time" --top-p 0.9 --temperature 0.8

## 7. Compare Pretraining Step Checkpoints

The training loop saves step checkpoints based on the `CHECKPOINT_EVERY_STEPS=50` value you used during training. If you look inside `artifacts/checkpoints/`, you should see files such as `dusty8m_step_300.pt`.

By default, the final checkpoint is saved as `dusty8m.pt` and used as the base model. However, the last checkpoint is not always the best one. Generate samples from a few step checkpoints, then pick the checkpoint with the best stability and rhythm before
the next step. Replace the step numbers below with checkpoints that exist in your `artifacts/checkpoints/` folder.

**Note:** In production, you would use an evaluation dataset to choose the best checkpoint. We skip that here to keep the workflow simple. For our tiny model and 100k TinyStories slice, step 300 produced the best lightweight demo result.

<details>
<summary>💡 What if I changed the batch size?</summary>

The suggested checkpoint steps assume the default Colab settings used above. If you changed `BATCH_SIZE`, dataset size, or `CHECKPOINT_EVERY_STEPS`, the same step number may not represent the same amount of training. Compare the checkpoint files that actually exist in `artifacts/checkpoints/` and promote the best one from your run.

</details>

In [ ]:
for step in [200, 250, 300]:
    print("=" * 80)
    print("CHECKPOINT_STEP:", step)
    !make generate PROFILE=dusty8m CHECKPOINT_STEP={step} PROMPT="deep in the forest, there lived a " TOP_P=0.9 TEMPERATURE=0.8
    # Windows fallback:
    # !uv run python dustylm/generate.py --profile dusty8m --checkpoint-step {step} --prompt "deep in the forest, there lived a " --top-p 0.9 --temperature 0.8

<details>
<summary>💡 Tip: Automated checkpoint evaluation</summary>

Testing multiple checkpoints by hand gets tedious. If you want to run a bulk evaluation across your training run, see [**Automated Checkpoint Comparison** in the advanced notebook](03_advanced_tools.ipynb).

</details>


---

## 8. Promote the Best Base Checkpoint

Choose the best checkpoint from Step 7 before we move to the next phase to teach the model its vacuum robot personality.

If a checkpoint performs better than the final one, it needs to be set as the default model by copying it to `artifacts/checkpoints/dusty8m.pt`.

The cell below automates this process for you. Set the `BEST_PRETRAIN_STEP` variable to your chosen checkpoint number, then run the cell to copy it to the default base checkpoint path.

In [ ]:
from pathlib import Path
import shutil

BEST_PRETRAIN_STEP = 300

src = Path(f"artifacts/checkpoints/dusty8m_step_{BEST_PRETRAIN_STEP}.pt")
dst = Path("artifacts/checkpoints/dusty8m.pt")
shutil.copy2(src, dst)

size_mb = dst.stat().st_size / (1024 * 1024)
print(f"✅ Step {BEST_PRETRAIN_STEP} successfully promoted to main base model!")
print(f"Copied {src} -> {dst} ({size_mb:.1f} MB)")

---
## 9. Fine-Tune with SFT

In this step, we teach the model its vacuum robot personality by training it on our synthetic dataset of user-robot conversations. Instead of starting from scratch, this phase builds on the base model you just set as the default. It already understands the English language, but it does not yet know how to carry on a structured chat.

SFT automatically initializes from the `artifacts/checkpoints/dusty8m.pt` base checkpoint defined by the `sft_dusty8m` profile.

> ⏱️ Two epochs of SFT take about 1 minute on a Google Colab T4 GPU. The SFT dataset has ~35K examples, so training is fast.


In [ ]:
%%time
# Google Colab T4 can run batch 224 comfortably.
# Reduce BATCH_SIZE=32 if running locally on an older laptop.
!make train-sft EPOCHS=2 CHECKPOINT_EVERY_STEPS=50 BATCH_SIZE=224
# Windows fallback, or if you want to see what make runs under the hood:
# !uv run python -m dustylm.train --profile sft_dusty8m --epochs 2 --checkpoint-every-steps 50 --batch-size 224

### Training Reference

Here are the reference numbers for the notebook's default run:

| Metric | Pretrain | SFT |
|---|---|---|
| Dataset examples | 100,000 | 33,820 |
| Batch size | 224 | 224 |
| Max seq len | 256 | 256 |
| Steps per epoch | ~382 | ~150 |
| Total steps run | 382 | ~300 |
| Best checkpoint | 300 | 200 |
| Tokens seen by full run | ~22M | ~16M |
| Epochs | 1 | 2 |

The promoted pretraining checkpoint is step 300, which has seen about 17M tokens. If you train with these settings, your results should be similar.


## 10. Compare SFT Step Checkpoints

`EPOCHS=2` produces ~300 steps and saves checkpoints every 50 steps. These checkpoints show early alignment, not final quality. In our run, the best Dusty persona checkpoint was around step 200.

<details>
<summary>💡 What if I changed the batch size?</summary>

The recommended SFT step assumes the default settings used above. If you changed `BATCH_SIZE`, evaluate the SFT checkpoints from your own run and promote the best one.

</details>

### 10.1 Generation Parameters

Small models have noisy token distributions, so generation settings matter:

| Parameter | Value | Effect |
|---|---|---|
| Temperature | 0.6 - 0.8 | Controls randomness. Lower is repetitive; higher can drift. |
| Top-K | 5 | Keeps only the 5 most likely next tokens. |
| Top-P | 0.8 - 0.9 | Keeps the smallest token set covering 80-90% probability mass. |

The `sft_dusty8m` profile sets these defaults in `dustylm/config.py`; the commands below override `TOP_P` and `TEMPERATURE` explicitly.

In [ ]:
# Sample checkpoints from the 2-epoch local run
for step in [150, 200, 250]:
    print("=" * 80)
    print("SFT CHECKPOINT_STEP:", step)
    !make generate PROFILE=sft_dusty8m CHECKPOINT_STEP={step} PROMPT="what makes you happy?" TOP_P=0.9 TEMPERATURE=0.6
    # Windows fallback:
    # !uv run python dustylm/generate.py --profile sft_dusty8m --checkpoint-step {step} --prompt "what makes you happy?" --top-p 0.9 --temperature 0.6

<details>
<summary>💡 Tip: Automated checkpoint evaluation</summary>

Testing multiple checkpoints by hand gets tedious. If you want to run a bulk evaluation across your training run, see [**Automated Checkpoint Comparison** in the advanced notebook](03_advanced_tools.ipynb).

</details>


### 10.2 What to Look For

Read the 2-epoch outputs as progress signals, not finished behavior:

1. **Step 150:** The model starts shifting toward Dusty's persona, though outputs are still chaotic.
2. **Step 200:** The chat format and Dusty persona are most balanced -- coherent identity, stable emotion, and the least overfitting.
3. **Step 250:** Responses appeared more repetitive and rigid in our run, which can be a sign that overfitting is starting.


> 💡 **Note:** The SFT dataset has ~35K examples; 2 epochs is enough for a convincing demo. If you train longer, watch for overfitting as the model memorizes responses instead of generalizing.

## 11. Promote the Best SFT Checkpoint

The chat cells load the `artifacts/checkpoints/dusty8m_sft.pt` checkpoint. Promotion just copies your chosen step checkpoint to that filename.

Since we ran a short 2-epoch demo, use step 200. The cell below copies that checkpoint to the default SFT checkpoint path.

In [ ]:
from pathlib import Path
import shutil

BEST_SFT_STEP = 200  # Replace this with your best step

src = Path(f"artifacts/checkpoints/dusty8m_sft_step_{BEST_SFT_STEP}.pt")
dst = Path("artifacts/checkpoints/dusty8m_sft.pt")
shutil.copy2(src, dst)

size_mb = dst.stat().st_size / (1024 * 1024)
print(f"✅ Step {BEST_SFT_STEP} successfully promoted to main SFT model!")
print(f"Copied {src} -> {dst} ({size_mb:.1f} MB)")

---
## 12. Chat with Your Trained Model

Run the cell below to test a few prompts with your promoted SFT checkpoint.

In [ ]:
from dustylm.inference import Inference

engine = Inference(
    checkpoint_path=REPO_ROOT / "artifacts" / "checkpoints" / "dusty8m_sft.pt",
    tokenizer_path=REPO_ROOT / "artifacts" / "tokenizers" / "dusty_tokenizer.json",
    device=device,
)


def chat(prompt):
    response = engine.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=64,
        temperature=0.6,
        top_p=0.8,
    )
    return response["choices"][0]["message"]["content"].strip()


for prompt in ["who are you?", "what makes you happy?"]:
    print(f"You> {prompt}")
    print(f"Dusty> {chat(prompt)}\n")

> **💡 Prefer the CLI?** If you want to chat in the terminal, open your system terminal outside of this notebook, navigate to the repository root, and run:
>
> ```bash
> make chat
> ```



---
## 13. Save Your Checkpoints

Cloud runtimes like Google Colab and RunPod are ephemeral. Your trained checkpoint files will be permanently deleted when the instance shuts down. Download them before closing this notebook.

The cell below downloads your promoted base and sft models. 
  - On **Google Colab** it triggers a browser download for each file. 
  - On **other cloud runtimes** (RunPod, etc.) it prints the file paths and sizes for manual download.



In [ ]:
import sys
from pathlib import Path

checkpoints_dir = Path("artifacts/checkpoints")
promoted = ["dusty8m.pt", "dusty8m_sft.pt"]

if "google.colab" in sys.modules:
    from google.colab import files

    for name in promoted:
        path = checkpoints_dir / name
        if path.exists():
            files.download(str(path))
    print("Downloads triggered. Check your browser downloads folder.")
else:
    for name in promoted:
        path = checkpoints_dir / name
        if path.exists():
            size = path.stat().st_size / 1024 / 1024
            print(f"{path}  {size:.1f} MB")
        else:
            print(f"{path}  not found")
    print(
        "\nDownload these files manually via your cloud provider's file browser or scp."
    )


---

## 14. What's Next?

Congratulations! 🚀 You have successfully trained an LLM from scratch, guided it through a pre-training phase to learn English, and fine-tuned it to adopt a specific persona.

If you want to push this model further, continue with the [Advanced Tools notebook](03_advanced_tools.ipynb), where we cover:

* **Changing the Persona:** How to edit the pipeline to make the model act like a different character entirely.

* **Synthetic Data Generation:** How to use API providers to generate your own custom SFT chat examples from scratch.

* **Temperature & Generation Tweaks:** How adjusting sampling parameters can fundamentally change the model's perceived personality.


---

## 15. Optional: Run from the Terminal

<details>
<summary>Step-by-step and automated commands</summary>

Now that you understand the mechanics, you can run the same pipeline directly from your terminal.

> **Local hardware:** The default commands use batch size 224, tuned for a Colab T4 GPU. If training is slow or runs out of memory locally, replace `BATCH_SIZE=224` with `BATCH_SIZE=32` in the step-by-step commands, or use the lower-memory automated example below. That automated configuration took about 15 minutes on an M1 Pro in our test. Changing the batch size also changes the checkpoint step numbers, so compare the checkpoints created by your own run.

### Step-by-step commands

```bash
make download-datasets
make tokenizer
make data-pretrain
make train-pretrain EPOCHS=1 CHECKPOINT_EVERY_STEPS=50 BATCH_SIZE=224
make data-sft
make train-sft EPOCHS=2 CHECKPOINT_EVERY_STEPS=50 BATCH_SIZE=224
make chat
```

### Automated end-to-end run

The guided notebook pauses for checkpoint comparison so you can observe how the model changes during pretraining and SFT. To run the same pipeline without stepping through notebook cells, use the automation command from the repository root:

```bash
# Default run, tuned for Colab T4 GPUs
make train-end-to-end

# Lower-memory run, measured at about 15 minutes on an M1 Pro
make train-end-to-end \
  E2E_FLAGS="--pretrain-batch-size 32 --sft-batch-size 32"
```

After the automated run finishes, chat with your trained model:

```bash
make chat
```

</details>
